In [1]:
# ============================================================
# CELDA 1
# Importar librerías
# ============================================================

import pandas as pd
import numpy as np
import re

# Mostrar todas las columnas cuando sea necesario
pd.set_option("display.max_columns", None)

# Evitar que los títulos largos se corten demasiado
pd.set_option("display.max_colwidth", 150)

In [2]:
# ============================================================
# CELDA 2
# Cargar la matriz de estudios
# ============================================================

archivo = "mtz-primarios-secundarios.csv"

# El archivo proporcionado usa:
# - separador ;
# - codificación latin-1
df = pd.read_csv(
    archivo,
    sep=";",
    encoding="latin-1"
)

print("Número total de estudios:", len(df))
print("Número de columnas:", len(df.columns))

df.head()

Número total de estudios: 87
Número de columnas: 42


,ID,Autores,Autores APA,Año,Título,País/Región,Fuente,Tipo de publicación,Dominio / contexto,Tipo de sistema,Etapa del ciclo de vida,Fuente de conocimiento,Tipo de conocimiento,Representación del conocimiento,Método computacional,Entrada,Salida,Actividad arquitectónica,Propiedad / objetivo de seguridad,Amenaza / vulnerabilidad,Mecanismo de verificación,Criterios / reglas verificadas,Diseño de evaluación,Dataset / casos de estudio,Línea base / comparación,Métricas,Resultados principales,Mecanismo de refinamiento,Tipo de retroalimentación,Trazabilidad,Explicabilidad,Limitaciones,Brechas / trabajo futuro,RQ1,RQ2,RQ3,RQ4,Calidad / observaciones,DOI / URL,Estado extracción,Completitud,Fecha extracción
0,1,Philipp Spelten; Morten-Christian Meyer; Anna Wagner; Klaus Wolf; Dirk Reith,Spelten et al.,2024,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,Alemania,"Information, 15, 21",Artículo de revista,Gestión de procesos y datos de simulación,Sistema de gestión semántica de datos de simulación / arquitectura de software basada en ontologías,"Diseño arquitectónico, integración y gestión de datos","Especificación industrial VMAP, ontologías de dominio, metadatos de simulación, rutinas de procesamiento y requisitos derivados de casos de uso","Conocimiento de dominio, estructural y procedimental","Ontologías, grafos de conocimiento, RDF/RDFS, OWL y relaciones semánticas","Ontology-Based Data Access (OBDA), población automática de grafos de conocimiento, consultas SPARQL y ejecución de rutinas semánticamente represen...","Archivos VMAP/HDF5, solicitudes de metadatos, ontologías y definiciones de rutinas","Grafo de conocimiento poblado, metadatos recuperados o generados y resultados derivados mediante rutinas de procesamiento","Diseño e implementación de una arquitectura de software para integración, consulta y procesamiento semántico de datos",NaN,NaN,Pruebas funcionales automatizadas del prototipo; no corresponde a verificación de seguridad,"Recuperación de metadatos, detección de duplicados, reutilización de metadatos existentes, ejecución de rutinas y generación de nuevos metadatos","Evaluación mediante prototipo y conjunto de pruebas funcionales, incluyendo un caso industrial de simulación",Archivos VMAP de prueba y caso de moldeo por soplado; parte de los datos industriales no está disponible públicamente,"Comparación de almacenamiento: VMAP 176 kB, STEP AP209 4278 kB y representación OWL 43 652 Kb",Tamaño de almacenamiento y cumplimiento de pruebas funcionales; no se reportan métricas comparativas amplias de rendimiento,"El prototipo automatiza la población del grafo, recupera metadatos mediante consultas semánticas, ejecuta rutinas para generar nuevos metadatos y ...",Bucle opcional de control de versiones para verificar actualización de metadatos y rutinas; refinamiento futuro de las ontologías,Verificación de vigencia/versión de metadatos y rutinas; no se implementa retroalimentación para refinamiento arquitectónico de seguridad,"Sí; relaciones explícitas entre archivos, datasets, metadatos, rutinas, entradas y salidas mediante ontologías y grafos de conocimiento","Parcial; la estructura semántica y las consultas SPARQL permiten inspeccionar relaciones y procedimientos, aunque la explicabilidad no se evalúa e...",Ontologías construidas manualmente; ausencia de un marco ontológico común ampliamente aceptado; prototipo aún no evaluado en explotación industria...,"Automatizar la generación de ontologías, alinear ontologías entre dominios, crear benchmarks SPDM/OBDA, evaluar en escenarios industriales y optim...",Sí,Parcial,No,No,"Evidencia relacionada/transferible. Aporta una implementación clara de conocimiento mediante ontologías y grafos, pero no aborda seguridad ni arqu...",10.3390/info15010021,Completa,100%,25/9/2026
1,2,Shanxin Sun; Fanchao Meng; Dianhui Chu,Sun et al.,2020,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,Chi

In [3]:
# ============================================================
# CELDA 3
# Revisar las columnas de la matriz
# ============================================================

for i, columna in enumerate(df.columns, start=1):
    print(f"{i:02d}. {columna}")

01. ID
02. Autores
03. Autores APA
04. Año
05. Título
06. País/Región
07. Fuente
08. Tipo de publicación
09. Dominio / contexto
10. Tipo de sistema
11. Etapa del ciclo de vida
12. Fuente de conocimiento
13. Tipo de conocimiento
14. Representación del conocimiento
15. Método computacional
16. Entrada
17. Salida
18. Actividad arquitectónica
19. Propiedad / objetivo de seguridad
20. Amenaza / vulnerabilidad
21. Mecanismo de verificación
22. Criterios / reglas verificadas
23. Diseño de evaluación
24. Dataset / casos de estudio
25. Línea base / comparación
26. Métricas
27. Resultados principales
28. Mecanismo de refinamiento
29. Tipo de retroalimentación
30. Trazabilidad
31. Explicabilidad
32. Limitaciones
33. Brechas / trabajo futuro
34. RQ1
35. RQ2
36. RQ3
37. RQ4
38. Calidad / observaciones
39. DOI / URL
40. Estado extracción
41. Completitud
42. Fecha extracción


In [4]:
# ============================================================
# CELDA 4
# Términos que caracterizan estudios secundarios
# ============================================================

patrones_secundarios = [
    r"systematic literature review",
    r"systematic review",
    r"systematic mapping study",
    r"\bmapping study\b",
    r"multivocal literature review",
    r"multivocal review",
    r"multivocal study",
    r"\bsurvey\b",
    r"scoping review",
    r"tertiary study",
    r"state[- ]of[- ]the[- ]art review",
    
    # Equivalentes en español
    r"revisión sistemática",
    r"revisión de literatura",
    r"revisión multivocal",
    r"mapeo sistemático",
    r"estudio de mapeo",
    r"artículo de revisión",
    r"revisión narrativa",
    r"revisión conceptual",
    
    # Variantes frecuentes
    r"comprehensive review",
    r"comprehensive survey"
]

patron_secundario = re.compile(
    "|".join(patrones_secundarios),
    flags=re.IGNORECASE
)

In [5]:
# ============================================================
# CELDA 5
# Construir texto para clasificación
# ============================================================

columnas_clasificacion = [
    "Título",
    "Tipo de publicación",
    "Diseño de evaluación"
]

def construir_texto(row):
    """
    Une las columnas relevantes de cada artículo.
    Los valores vacíos se convierten en cadena vacía.
    """
    
    valores = []
    
    for columna in columnas_clasificacion:
        valor = row.get(columna, "")
        
        if pd.notna(valor):
            valores.append(str(valor))
    
    return " | ".join(valores)


df["_texto_clasificacion"] = df.apply(
    construir_texto,
    axis=1
)

df[
    ["ID", "Título", "_texto_clasificacion"]
].head()

,ID,Título,_texto_clasificacion
0,1,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses | Artículo de revista | E...
1,2,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,A Model Driven Approach to Constructing Knowledge Graph from Relational Database | Artículo de conferencia | Caso demostrativo de transformación d...
2,3,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,A model-driven knowledge graph treatment for process-centric cyber threat mitigation | Artículo de revista | Design Science Research + AMME; demos...
3,4,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support","Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support | Artículo de revista | 50 consultas en..."
4,5,An Application Security Requirements Analysis Method Based on Knowledge Graph,An Application Security Requirements Analysis Method Based on Knowledge Graph | Artículo de conferencia | No se reporta una evaluación experimenta...


In [6]:
# ============================================================
# CELDA 6
# Clasificar estudios
# ============================================================

def clasificar_estudio(row):
    """
    Clasificación operacional para la RSL.

    SECUNDARIO:
        Si sintetiza investigación previa:
        SLR, survey, mapping study, multivocal review, etc.

    PRIMARIO:
        Si reporta una contribución original:
        método, framework, algoritmo, arquitectura,
        experimento, estudio de caso, herramienta, etc.
    """
    
    texto = row["_texto_clasificacion"]
    
    if patron_secundario.search(texto):
        return "Secundario"
    
    return "Primario"


df["Grupo"] = df.apply(
    clasificar_estudio,
    axis=1
)

df[
    ["ID", "Título", "Tipo de publicación", "Grupo"]
].head(10)

,ID,Título,Tipo de publicación,Grupo
0,1,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,Artículo de revista,Primario
1,2,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,Artículo de conferencia,Primario
2,3,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,Artículo de revista,Primario
3,4,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support",Artículo de revista,Primario
4,5,An Application Security Requirements Analysis Method Based on Knowledge Graph,Artículo de conferencia,Primario
5,6,An Empirical Evaluation of Large Language Models Applying Software Architectural Patterns,Artículo de revista,Primario
6,7,An Ontology for Privacy Requirements via a Systematic Literature Review,Artículo de revista; investigación conceptual/metodológica basada en SLR,Secundario
7,8,An ontology-based learning approach for automatically classifying security requirements,Artículo de revista; investigación empírica/experimental,Primario
8,9,An ontology-based secure design framework for graph-based databases,Artículo de revista; propuesta de framework con implementación y estudio de caso,Primario
9,10,Automated regulatory compliance for AI systems in the security domain: The case of dual-use deployment,Artículo de Revista,Primario


In [7]:
# ============================================================
# CELDA 7
# Resumen de clasificación
# ============================================================

resumen = (
    df["Grupo"]
    .value_counts()
    .rename_axis("Grupo")
    .reset_index(name="Cantidad")
)

resumen

,Grupo,Cantidad
0,Primario,74
1,Secundario,13


In [8]:
# ============================================================
# CELDA 8
# Estudios detectados como secundarios
# ============================================================

secundarios = (
    df[df["Grupo"] == "Secundario"]
    .copy()
)

secundarios[
    [
        "ID",
        "Año",
        "Título",
        "Tipo de publicación",
        "Diseño de evaluación"
    ]
].sort_values("ID")

,ID,Año,Título,Tipo de publicación,Diseño de evaluación
6,7,2021,An Ontology for Privacy Requirements via a Systematic Literature Review,Artículo de revista; investigación conceptual/metodológica basada en SLR,SLR en dos fases + análisis de cobertura + consolidación; comparación con ontologías existentes; demostración sobre caso AAL; validación mediante ...
14,15,2026,"Generative AI for software architecture. Applications, challenges, and future directions",Revisión multivocal de literatura,Multivocal literature review
22,23,2025,Multivocal study on microservice dependencies,Revisión multivocal de literatura,"MLR con protocolo, QA y extracción sistemática"
29,30,2024,Shielding software systems: A comparison of security by design and privacy by design based on a systematic literature review,Revisión sistemática de literatura,Revisión sistemática siguiendo PRISMA; codificación y análisis narrativo
39,40,2025,A Survey of Retrieval-Augmented Generation (RAG) for Large Language Models,Artículo de revisión/survey,Survey/revisión conceptual
41,42,2025,Can LLMs Be Trusted for Evaluating RAG Systems? A Survey of Methods and Datasets,Revisión sistemática/survey,SLR de 63 artículos
53,54,2025,Integrating External Knowledge with LLMs: A Systematic Review of RAG Approaches,Revisión sistemática/survey,Revisión sistemática/estado del arte
63,64,2025,RAG augmented Large Language Models as a tool for Education,Artículo de conferencia; survey/propuesta de chatbot educativo,Survey/revisión narrativa; propuesta para exploración futura
64,65,2025,"RAG Powered LLMs for QA: Evolution, Challenges, Applications, and Future Directions",Artículo de revisión/survey,Survey/revisión
69,70,2025,"Retrieval-Augmented Generation: A Comprehensive Survey of Foundations, Evolution, Applications, Limitations, and Future Directions",Artículo de revisión/survey,Revisión/survey


In [9]:
# ============================================================
# CELDA 9
# Estudios detectados como primarios
# ============================================================

primarios = (
    df[df["Grupo"] == "Primario"]
    .copy()
)

primarios[
    [
        "ID",
        "Año",
        "Título",
        "Tipo de publicación",
        "Diseño de evaluación"
    ]
].sort_values("ID")

,ID,Año,Título,Tipo de publicación,Diseño de evaluación
0,1,2024,A Methodology for Integrating Hierarchical VMAP-Data Structures into an Ontology Using Semantically Represented Analyses,Artículo de revista,"Evaluación mediante prototipo y conjunto de pruebas funcionales, incluyendo un caso industrial de simulación"
1,2,2020,A Model Driven Approach to Constructing Knowledge Graph from Relational Database,Artículo de conferencia,Caso demostrativo de transformación de MySQL a Neo4j
2,3,2026,A model-driven knowledge graph treatment for process-centric cyber threat mitigation,Artículo de revista,"Design Science Research + AMME; demostración y evaluación mediante escenarios controlados, consultas semánticas y dos experimentos RAG/GraphRAG"
3,4,2025,"Aligning DevOps and Microservice Architecture: Empirical Mapping, Taxonomy, and RAG-Based Decision Support",Artículo de revista,"50 consultas en cinco categorías, doble anotación humana, comparación con BM25 y recuperación basada únicamente en embeddings, más estudios indust..."
4,5,2023,An Application Security Requirements Analysis Method Based on Knowledge Graph,Artículo de conferencia,No se reporta una evaluación experimental formal
...,...,...,...,...,...
80,81,2025,Towards Intelligent Enterprise Assistants: Leveraging RAG to Connect LLMs with Structured Business Data,Artículo de conferencia; RAG para datos estructurados/NL2SQL,Experimento en schemas/queries de benchmark
82,83,2024,Trace Anomaly Detection for Microservice Systems via Graph-based Semi-supervised Learning,Artículo de conferencia; graph-based semi-supervised anomaly detection,"Benchmark TrainTicket con ChaosMesh, partición train/validation/test y baselines"
83,84,2025,TrioXpert: An Automated Incident Management Framework for Microservice System,Artículo de conferencia; LLM multi-expert incident management,Dos datasets + seis baselines + ablations + despliegue Lenovo
85,86,2024,VAE-based Fault Diagnosis for Microservice System,Artículo de conferencia; unsupervised graph/VAE anomaly detection,Experimentos en dos datasets de microservicios


In [10]:
# ============================================================
# CELDA 10
# Detectar artículos conceptuales o híbridos
# que deberían revisarse manualmente
# ============================================================

patrones_revision_manual = [
    r"conceptual",
    r"perspectiva",
    r"artículo de visión",
    r"proposal",
    r"propuesta arquitectónica",
    r"validación futura",
    r"no se reporta una evaluación experimental",
    r"literature-based",
    r"arquitectura conceptual"
]

patron_revision = re.compile(
    "|".join(patrones_revision_manual),
    flags=re.IGNORECASE
)


def requiere_revision_manual(row):
    
    texto = row["_texto_clasificacion"]
    
    return bool(patron_revision.search(texto))


df["Revision_manual"] = df.apply(
    requiere_revision_manual,
    axis=1
)

dudosos = df[
    (df["Revision_manual"] == True)
    & (df["Grupo"] == "Primario")
].copy()

dudosos[
    [
        "ID",
        "Título",
        "Tipo de publicación",
        "Diseño de evaluación",
        "Grupo"
    ]
].sort_values("ID")

,ID,Título,Tipo de publicación,Diseño de evaluación,Grupo
4,5,An Application Security Requirements Analysis Method Based on Knowledge Graph,Artículo de conferencia,No se reporta una evaluación experimental formal,Primario
11,12,BridgeSec: Facilitating effective communication between security engineering and systems engineering,Artículo de revista; framework conceptual con prototipo open-source,Validación técnica mediante prototipo; dos escenarios realistas ilustrativos; comparación cualitativa con trabajos relacionados,Primario
19,20,"LLM experimentation through knowledge graphs: Towards improved management, repeatability, and verification",Artículo de revista; arquitectura/testbed conceptual con prototipo,Caso de nutrición con 100 prompts ×10 repeticiones en dos LLMs,Primario
30,31,Towards Trustworthy AI Software Development Assistance,Artículo de visión / New Ideas and Emerging Results,Artículo conceptual: propone planes de evaluación para cada componente; no implementación integrada aún,Primario
34,35,Agentic RAG with Human-in-the-Retrieval,Artículo de conferencia; propuesta arquitectónica/conceptual,Propuesta conceptual y discusión de escenarios/desafíos de diseño; evaluación empírica amplia NR,Primario
35,36,A Global Hierarchical LLM Framework for Precise Responses through Subscriptions to Country LLMs,Artículo de conferencia; framework conceptual,Framework conceptual; simulación/validación futura propuesta,Primario
47,48,Exploring the Application of Retrieval-Augmented Generation Technology in Defense Technology Intelligence,Artículo de conferencia; análisis/aplicación conceptual,Análisis conceptual/literature-based de escenarios y deployment,Primario
62,63,"RAG4DS: Retrieval-Augmented Generation for Data SpacesA Unified Lifecycle, Challenges, and Opportunities",Artículo de revista; arquitectura/lifecycle conceptual,Arquitectura conceptual + dos use cases,Primario
79,80,The Rise and Design of Enterprise Large Language Models,Artículo de perspectiva/AI in Industry,Artículo conceptual con ejemplos/discusiones de industria,Primario


In [11]:
# ============================================================ 
# CELDA 11
# Crear clasificación final
# ============================================================

# Partimos de la clasificación automática
df["Grupo_final"] = df["Grupo"]

# Después de revisar los casos dudosos,
# se pueden cambiar IDs específicos manualmente.

In [12]:
# EJEMPLO
# Sustituir estos IDs por los determinados
# después de la lectura y aplicación del protocolo.

ids_secundarios_manual = [
    # 31,
    # 35,
    # 36
    5,12,20,31,35,36,48,63,80
]

df.loc[
    df["ID"].isin(ids_secundarios_manual),
    "Grupo_final"
] = "Secundario"

In [13]:
# EJEMPLO:
# artículos que el investigador determina que son primarios.

ids_primarios_manual = [
    # 7
]

df.loc[
    df["ID"].isin(ids_primarios_manual),
    "Grupo_final"
] = "Primario"

In [14]:
# ============================================================
# CELDA 12
# Verificar clasificación final
# ============================================================

resumen_final = (
    df["Grupo_final"]
    .value_counts()
    .rename_axis("Grupo")
    .reset_index(name="Cantidad")
)

resumen_final

,Grupo,Cantidad
0,Primario,65
1,Secundario,22


In [15]:
# Verificación esperada de estudios primarios

PRIMARIOS_ESPERADOS = 65

n_primarios = (
    df["Grupo_final"] == "Primario"
).sum()

n_secundarios = (
    df["Grupo_final"] == "Secundario"
).sum()

print("Primarios:", n_primarios)
print("Secundarios:", n_secundarios)
print("Total:", n_primarios + n_secundarios)

if n_primarios == PRIMARIOS_ESPERADOS:
    print("\n✓ La cantidad de estudios primarios es correcta.")
else:
    print(
        f"\n⚠ Se esperaban {PRIMARIOS_ESPERADOS} estudios primarios, "
        f"pero actualmente existen {n_primarios}."
    )
    print("Revise los estudios marcados como conceptuales/híbridos.")

Primarios: 65
Secundarios: 22
Total: 87

✓ La cantidad de estudios primarios es correcta.


In [16]:
# ============================================================
# CELDA 13
# Separar en dos DataFrames definitivos
# ============================================================

primarios = (
    df[df["Grupo_final"] == "Primario"]
    .copy()
)

secundarios = (
    df[df["Grupo_final"] == "Secundario"]
    .copy()
)

print("Estudios primarios:", len(primarios))
print("Estudios secundarios:", len(secundarios))
print("Total:", len(primarios) + len(secundarios))

Estudios primarios: 65
Estudios secundarios: 22
Total: 87


In [17]:
# ============================================================
# CELDA 14
# Limpiar columnas usadas solamente por el algoritmo
# ============================================================

columnas_auxiliares = [
    "_texto_clasificacion"
]

primarios = primarios.drop(
    columns=columnas_auxiliares,
    errors="ignore"
)

secundarios = secundarios.drop(
    columns=columnas_auxiliares,
    errors="ignore"
)

In [18]:
# ============================================================
# CELDA 15
# Exportar resultados
# ============================================================

primarios.to_csv(
    "estudios_primarios.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

secundarios.to_csv(
    "estudios_secundarios.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Archivos generados:")
print(" - estudios_primarios.csv")
print(" - estudios_secundarios.csv")

Archivos generados:
 - estudios_primarios.csv
 - estudios_secundarios.csv


In [19]:
# ============================================================
# CELDA 16
# Exportar ambos grupos en un solo Excel
# ============================================================

with pd.ExcelWriter(
    "clasificacion_estudios_RSL.xlsx",
    engine="openpyxl"
) as writer:
    
    primarios.to_excel(
        writer,
        sheet_name="Primarios",
        index=False
    )
    
    secundarios.to_excel(
        writer,
        sheet_name="Secundarios",
        index=False
    )
    
    df.to_excel(
        writer,
        sheet_name="Clasificacion_completa",
        index=False
    )

print("Archivo generado: clasificacion_estudios_RSL.xlsx")

Archivo generado: clasificacion_estudios_RSL.xlsx
